In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Paths
PROCESSED = '../data/processed/'

# Load master dataset
df = pd.read_csv(PROCESSED + 'master_dataset.csv')

print("Data loaded!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample:\n{df.head()}")

Data loaded!
Shape: (30, 10)

Columns: ['year', 'trade_value', 'avg_tariff_rate', 'unemployment_rate', 'exchange_rate_mxn', 'exchange_rate_cad', 'cpi', 'gdp_growth', 'country_Canada', 'country_Mexico']

Sample:
   year  trade_value  avg_tariff_rate  unemployment_rate  exchange_rate_mxn  \
0  2020   270.025524             9.14           7.725000          21.546209   
1  2021   357.274663             9.18           5.633333          20.284383   
2  2024   411.886683             9.28           4.116667          18.306223   
3  2011   315.324753             9.35           8.033333          12.427028   
4  2019   318.588850             9.08           3.525000          19.246934   

   exchange_rate_cad         cpi  gdp_growth  country_Canada  country_Mexico  
0           1.342203  258.855750       1.575            True           False  
1           1.253339  270.973417       5.750            True           False  
2           1.369870  313.698167       2.400            True           False 

In [3]:
# ── Sort data properly ─────────────────────────────
df = df.sort_values(['country_Canada', 'country_Mexico', 'year']).reset_index(drop=True)

# ── Lag Features (previous year's trade value) ─────
df['trade_value_lag1'] = df.groupby(['country_Canada'])['trade_value'].shift(1)
df['trade_value_lag2'] = df.groupby(['country_Canada'])['trade_value'].shift(2)

# ── Rolling Average (3-year moving average) ────────
df['trade_value_rolling3'] = df.groupby(['country_Canada'])['trade_value'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# ── Year-over-Year Growth Rate ─────────────────────
df['trade_growth_pct'] = df.groupby(['country_Canada'])['trade_value'].pct_change() * 100

print("Lag features created!")
print(df[['year', 'trade_value', 'trade_value_lag1', 
          'trade_value_lag2', 'trade_value_rolling3', 
          'trade_growth_pct']].head(10))

Lag features created!
   year  trade_value  trade_value_lag1  trade_value_lag2  \
0  2010   229.985623               NaN               NaN   
1  2011   262.873596        229.985623               NaN   
2  2012   277.593640        262.873596        229.985623   
3  2013   280.556040        277.593640        262.873596   
4  2014   295.729963        280.556040        277.593640   
5  2015   296.433325        295.729963        280.556040   
6  2016   293.500607        296.433325        295.729963   
7  2017   312.666740        293.500607        296.433325   
8  2018   343.680541        312.666740        293.500607   
9  2019   356.093584        343.680541        312.666740   

   trade_value_rolling3  trade_growth_pct  
0            229.985623               NaN  
1            246.429609         14.300013  
2            256.817620          5.599666  
3            273.674425          1.067172  
4            284.626548          5.408518  
5            290.906443          0.237839  
6        

In [8]:
# ── Tariff Change Features ─────────────────────────
df['tariff_change'] = df.groupby(['country_Canada'])['avg_tariff_rate'].diff()
df['tariff_change_pct'] = df.groupby(['country_Canada'])['avg_tariff_rate'].pct_change() * 100

# ── CPI Change (Inflation Rate) ────────────────────
df['cpi_change'] = df.groupby(['country_Canada'])['cpi'].pct_change() * 100

# ── Exchange Rate Volatility ───────────────────────
df['mxn_change'] = df.groupby(['country_Canada'])['exchange_rate_mxn'].pct_change() * 100
df['cad_change'] = df.groupby(['country_Canada'])['exchange_rate_cad'].pct_change() * 100

# ── Trade Acceleration (change in growth rate) ─────
df['trade_acceleration'] = df.groupby(['country_Canada'])['trade_growth_pct'].diff()

print("Economic change features created!")
print(df[['year', 'tariff_change', 'cpi_change', 
          'mxn_change', 'trade_acceleration']].head(10))
print(f"\nTotal features so far: {df.shape[1]}")

Economic change features created!
   year  tariff_change  cpi_change  mxn_change  trade_acceleration
0  2010            NaN         NaN         NaN                 NaN
1  2011          -0.07    3.139652   -1.557707                 NaN
2  2012          -0.07    2.073191    5.849059           -8.700346
3  2013          -0.08    1.465972   -3.006427           -4.532494
4  2014          -0.07    1.615463    4.261942            4.341346
5  2015          -0.03    0.121137   19.329659           -5.170678
6  2016          -0.02    1.267361   17.601300           -1.227174
7  2017          -0.03    2.131445    1.160727            7.519520
8  2018          -0.03    2.439000    1.767813            3.388938
9  2019           0.06    1.813259    0.151072           -6.307326

Total features so far: 24


In [9]:
# ── Interaction Features ───────────────────────────
# Tariff burden adjusted for inflation
df['tariff_cpi_interaction'] = df['avg_tariff_rate'] * df['cpi']

# Economic pressure index
df['economic_pressure'] = df['unemployment_rate'] * df['avg_tariff_rate']

# Currency-adjusted tariff impact
df['tariff_mxn_interaction'] = df['avg_tariff_rate'] * df['exchange_rate_mxn']
df['tariff_cad_interaction'] = df['avg_tariff_rate'] * df['exchange_rate_cad']

print("Interaction features created!")
print(df[['year', 'tariff_cpi_interaction', 
          'economic_pressure', 'tariff_mxn_interaction']].head(8))
print(f"\nTotal columns: {df.shape[1]}")

Interaction features created!
   year  tariff_cpi_interaction  economic_pressure  tariff_mxn_interaction
0  2010             2054.277490          77.715000              118.914949
1  2011             2103.030050          75.111667              116.192711
2  2012             2130.558853          62.253333              122.068118
3  2013             2143.156100          58.113333              117.377555
4  2014             2161.207950          47.323833              121.448965
5  2015             2156.715925          40.646667              144.448433
6  2016             2179.249183          42.373333              169.499887
7  2017             2218.345050          39.442917              170.900796

Total columns: 24


In [7]:
# ── Drop rows with NaN from lag features ──────────
df_features = df.dropna().reset_index(drop=True)

# ── Save to processed folder ───────────────────────
df_features.to_csv(PROCESSED + 'features_dataset.csv', index=False)

print("Features dataset saved!")
print(f"Shape after dropping NaN: {df_features.shape}")
print(f"\nAll features: {list(df_features.columns)}")

Features dataset saved!
Shape after dropping NaN: (26, 24)

All features: ['year', 'trade_value', 'avg_tariff_rate', 'unemployment_rate', 'exchange_rate_mxn', 'exchange_rate_cad', 'cpi', 'gdp_growth', 'country_Canada', 'country_Mexico', 'trade_value_lag1', 'trade_value_lag2', 'trade_value_rolling3', 'trade_growth_pct', 'tariff_change', 'tariff_change_pct', 'cpi_change', 'mxn_change', 'cad_change', 'trade_acceleration', 'tariff_cpi_interaction', 'economic_pressure', 'tariff_mxn_interaction', 'tariff_cad_interaction']
